### Fraud Detection System

#### Import important libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data=pd.read_csv(r"../data/raw/data.csv")
data

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0


#### Removing Null values

In [3]:
print("Total null values:",data.isnull().sum().sum())

Total null values: 0


#### Removing duplicates

In [4]:
print("Total duplicates:",data.duplicated().sum())

Total duplicates: 0


#### Drop unwanted columns

In [5]:
data=data.drop(columns="isFlaggedFraud")

#### Encoding

In [6]:
# Encoding of payment type
from sklearn.preprocessing import  LabelEncoder
type_le=LabelEncoder()
type_le.fit(data["type"])
data["type"] = type_le.transform(data["type"])
print(type_le.classes_)

['CASH_IN' 'CASH_OUT' 'DEBIT' 'PAYMENT' 'TRANSFER']


In [7]:
# Feature Engineering
data["nameOrig"]=data["nameOrig"].str[0]
data["nameDest"]=data["nameDest"].str[0]


In [8]:
# Encoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
col=["nameOrig","nameDest"]
for i in col:
    dest_le=LabelEncoder()
    dest_le.fit(data[i])
    data[i]=dest_le.transform(data[i])
data

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,1,3,9839.64,0,170136.00,160296.36,1,0.00,0.00,0
1,1,3,1864.28,0,21249.00,19384.72,1,0.00,0.00,0
2,1,4,181.00,0,181.00,0.00,0,0.00,0.00,1
3,1,1,181.00,0,181.00,0.00,0,21182.00,0.00,1
4,1,3,11668.14,0,41554.00,29885.86,1,0.00,0.00,0
...,...,...,...,...,...,...,...,...,...,...
6362615,743,1,339682.13,0,339682.13,0.00,0,0.00,339682.13,1
6362616,743,4,6311409.28,0,6311409.28,0.00,0,0.00,0.00,1
6362617,743,1,6311409.28,0,6311409.28,0.00,0,68488.84,6379898.11,1
6362618,743,4,850002.52,0,850002.52,0.00,0,0.00,0.00,1


#### Feature Engineering

In [9]:
data["day"]=data["step"]//24
data["hour"]=data["step"]%24
data.drop(columns="step",inplace=True)

#### Feature Scaling

In [10]:
from sklearn.preprocessing import MinMaxScaler
sc_col=["amount","oldbalanceOrg","newbalanceOrig","oldbalanceDest","newbalanceDest"]
for i in sc_col:
    ms=MinMaxScaler()
    ms.fit(data[[i]])
    data[[i]]=ms.transform(data[[i]])
data

,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,day,hour
0,3,0.000106,0,0.002855,0.003233,1,0.000000,0.000000,0,0,1
1,3,0.000020,0,0.000357,0.000391,1,0.000000,0.000000,0,0,1
2,4,0.000002,0,0.000003,0.000000,0,0.000000,0.000000,1,0,1
3,1,0.000002,0,0.000003,0.000000,0,0.000059,0.000000,1,0,1
4,3,0.000126,0,0.000697,0.000603,1,0.000000,0.000000,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
6362615,1,0.003674,0,0.005701,0.000000,0,0.000000,0.000954,1,30,23
6362616,4,0.068272,0,0.105923,0.000000,0,0.000000,0.000000,1,30,23
6362617,1,0.068272,0,0.105923,0.000000,0,0.000192,0.017912,1,30,23
6362618,4,0.009195,0,0.014265,0.000000,0,0.000000,0.000000,1,30,23


#### Split Data

In [11]:
from sklearn.model_selection import train_test_split
col=["type","amount","nameOrig","oldbalanceOrg","newbalanceOrig","nameDest","oldbalanceDest","newbalanceDest","day","hour"]
x=data[col]
y=data["isFraud"]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

#### Model selection

##### Logistic Regression

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score,precision_score,f1_score,r2_score
lr=LogisticRegression()
lr.fit(x_train,y_train)
print(lr.score(x_train,y_train)*100,lr.score(x_test,y_test)*100)
lr_pred=lr.predict(x_test)

lr_recall_score=recall_score(y_test,lr_pred)
lr_precision_score=precision_score(y_test,lr_pred)
lr_f1_score=f1_score(y_test,lr_pred)
lr_acc=lr.score(x_test,y_test)*100

99.87047395569749 99.87269395311993


c:\Users\Vaibhav\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


##### Decision Tree

In [13]:
from sklearn.tree import DecisionTreeClassifier

dt=DecisionTreeClassifier()
dt.fit(x_train,y_train)
print(dt.score(x_train,y_train)*100,dt.score(x_test,y_test)*100)
dt_pred=dt.predict(x_test)

dt_recall_score=recall_score(y_test,dt_pred)
dt_precision_score=precision_score(y_test,dt_pred)
dt_f1_score=f1_score(y_test,dt_pred)
dt_acc=dt.score(x_test,y_test)*100


100.0 99.97037383970753


##### Naive Bayes

In [14]:
from sklearn.naive_bayes import GaussianNB,BernoulliNB
gb=GaussianNB()
gb.fit(x_train,y_train)
print(gb.score(x_train,y_train)*100,gb.score(x_test,y_test)*100)
gb_pred=gb.predict(x_test)

gb_recall_score=recall_score(y_test,gb_pred)
gb_precision_score=precision_score(y_test,gb_pred)
gb_f1_score=f1_score(y_test,gb_pred)
gb_acc=gb.score(x_test,y_test)*100


91.2727579204793 91.30161788697109


In [15]:
bn=BernoulliNB()
bn.fit(x_train,y_train)
print(bn.score(x_train,y_train)*100,bn.score(x_test,y_test)*100)
bn_pred=bn.predict(x_test)

bn_recall_score=recall_score(y_test,bn_pred)
bn_precision_score=precision_score(y_test,bn_pred)
bn_f1_score=f1_score(y_test,bn_pred)
bn_acc=bn.score(x_test,y_test)*100

99.87057218567193 99.8727725370995


##### Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier
rd = RandomForestClassifier(
    n_estimators=20,
    n_jobs=-1,
    random_state=42
)
rd.fit(x_train,y_train)
print(rd.score(x_train,y_train)*100,rd.score(x_test,y_test)*100)
rd_pred=rd.predict(x_test)

rd_recall_score=recall_score(y_test,rd_pred)
rd_precision_score=precision_score(y_test,rd_pred)
rd_f1_score=f1_score(y_test,rd_pred)
rd_acc=rd.score(x_test,y_test)*100

99.99819256847022 99.97768214980621


#### Model Selection

In [17]:
comparison_table={
"Model":["Logistic Regression","Decision Tree","Gaussian NB","Bernoulli NB","Random Forest"],
"Recall":[lr_recall_score,dt_recall_score,gb_recall_score,bn_recall_score,rd_recall_score],
"Precision":[lr_precision_score,dt_precision_score,gb_precision_score,bn_precision_score,rd_precision_score],
"F1 Score":[lr_f1_score,dt_f1_score,gb_f1_score,bn_f1_score,rd_f1_score],
"Accuracy":[lr_acc,dt_acc,gb_acc,bn_acc,rd_acc]
}
comparison_table=pd.DataFrame(comparison_table)
comparison_table

,Model,Recall,Precision,F1 Score,Accuracy
0,Logistic Regression,0.000000,0.000000,0.000000,99.872694
1,Decision Tree,0.885802,0.881991,0.883893,99.970374
2,Gaussian NB,0.775926,0.011265,0.022208,91.301618
3,Bernoulli NB,0.000617,1.000000,0.001234,99.872773
4,Random Forest,0.836420,0.986172,0.905144,99.977682


#### Save comparison table 

In [18]:
comparison_table.to_csv(r"../data/processed/Models_comparison.csv",index=False)

#### Hyperparamter Tuning

In [19]:
new_data=data.sample(n=200000,random_state=42)

new_data_x=new_data.drop(columns=["isFraud"])
new_data_y=new_data["isFraud"]


In [20]:
from sklearn.model_selection import RandomizedSearchCV
li={"criterion":['gini', 'entropy', 'log_loss'],"splitter":['best', 'random'],"max_depth":[i for i in range(1,26)],"random_state":[10,20,30,40,42,100]}
rs=RandomizedSearchCV(dt,param_distributions=li,cv=5,n_iter=10)
rs.fit(new_data_x,new_data_y)
rs.best_params_,rs.best_score_

({'splitter': 'best',
  'random_state': 40,
  'max_depth': 8,
  'criterion': 'log_loss'},
 np.float64(0.9995800000000001))

#### Final Model

In [21]:
dt=DecisionTreeClassifier(splitter="best",random_state=20,max_depth=8,criterion="entropy")
dt.fit(x_train,y_train)
print(dt.score(x_train,y_train)*100,dt.score(x_test,y_test)*100)

99.96728941850999 99.96707331256621


#### Export Model

In [23]:
import joblib

joblib.dump(dt, "../Models/dt_model.pkl")
joblib.dump(ms,"../Models/dt_scaler.pkl")
joblib.dump(type_le,"../Models/dt_type_encoder.pkl")
joblib.dump(dest_le,"../Models/dt_dest_encoder.pkl")
joblib.dump(gb, "../Models/gb_model.pkl")
joblib.dump(rd, "../Models/rf_model.pkl")

['../Models/rf_model.pkl']